# 02 · Nhãn, mẫu hiệu dụng và sức mạnh kiểm định
Câu hỏi: với ~1.700 phiên và nhãn 5 phiên chồng lấn, loại mô hình nào chứng minh được là có thể?

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from _common import *
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3})
news, px, tg = load()
vn30 = px[px.ticker == 'VN30'].set_index('date').sort_index()
print(f'news {len(news):,} | price rows {len(px):,} | targets {len(tg):,}')

## 1. Cân bằng nhãn

In [ ]:
tg['year'] = tg.date.dt.year
print(tg.groupby('year')[['y_up', 'y_drop']].mean().round(3).assign(n=tg.groupby('year').size()))
print('tổng:', tg[['y_up', 'y_drop']].mean().round(3).to_dict())

## 2. Tự tương quan: chồng lấn làm giảm cỡ mẫu

In [ ]:
from statsmodels.tsa.stattools import acf
r1 = np.log(vn30.close).diff().dropna()
f5 = tg.set_index('date').fwd_ret
a1, a5 = acf(r1, nlags=10), acf(f5, nlags=10)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].bar(range(11), a1); ax[0].set_title('ACF lợi suất 1 phiên'); ax[1].bar(range(11), a5); ax[1].set_title('ACF lợi suất 5 phiên chồng lấn')
for a in ax: a.axhline(1.96/np.sqrt(len(r1)), color='r', ls='--'); a.axhline(-1.96/np.sqrt(len(r1)), color='r', ls='--')
savefig(fig, 'nb02_acf'); plt.show()
# Cỡ mẫu hiệu dụng cho nhãn chồng lấn: n / (1 + 2*sum Bartlett*rho_k) với K=4
K = 4; rho = acf(f5, nlags=K)[1:]; w = 1 - np.arange(1, K + 1) / (K + 1)
n = len(f5); inflation = 1 + 2 * np.sum(w * rho)
print(f'n={n}, hệ số phồng phương sai={inflation:.2f}, n_eff≈{n/inflation:.0f}; cửa sổ không chồng lấn={n//5}')
for col in ['y_up', 'y_drop']:
    ac = acf(tg[col], nlags=6)[1:]; print(col, 'ACF lag1..5:', ac[:5].round(2))

## 3. Sức mạnh kiểm định: tương quan nhỏ nhất phát hiện được

In [ ]:
from scipy import stats
def min_detectable_r(n_eff, alpha=.05, power=.8, one_sided=True):
    za = stats.norm.ppf(1 - alpha / (1 if one_sided else 2)); zb = stats.norm.ppf(power)
    return np.tanh((za + zb) / np.sqrt(n_eff - 3))
rows = []
for label, ne in [('mẫu độc lập 5 phiên (~%d)' % (n//5), n//5), ('n_eff (hiệu chỉnh ACF)', int(n/inflation)), ('hậu mốc LLM: 2026 (~34)', 34)]:
    rows.append((label, ne, min_detectable_r(ne)))
pw = pd.DataFrame(rows, columns=['tình huống', 'n_eff', 'r_min (80% power, 1 phía, α=5%)']); print(pw.round(3).to_string(index=False))
ns = np.arange(30, 1800, 10); fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ns, [min_detectable_r(x) for x in ns]); ax.set_xlabel('n hiệu dụng'); ax.set_ylabel('r tối thiểu'); ax.set_title('Hiệu ứng nhỏ nhất phát hiện được')
savefig(fig, 'nb02_power'); plt.show()

## 4. Tính dừng/chế độ

In [ ]:
from statsmodels.tsa.stattools import adfuller
print('ADF p (lợi suất 1 phiên):', round(adfuller(r1)[1], 4))
vol_y = r1.groupby(r1.index.year).std() * np.sqrt(252)
fig, ax = plt.subplots(figsize=(7, 3)); vol_y.plot.bar(ax=ax); ax.set_title('Biến động annualised theo năm')
savefig(fig, 'nb02_vol_year'); plt.show()
print('Tỷ lệ biến động lớn nhất/nhỏ nhất theo năm:', round(vol_y.max() / vol_y.min(), 2))

## Kết luận từ notebook 02
- Lợi suất 5 phiên chồng lấn có tự tương quan lớn, nên n hiệu dụng gần số cửa sổ không chồng lấn, không phải 1.700.
- Hiệu ứng nhỏ nhất phát hiện được (r_min) cao với n hiệu dụng này; chỉ những tín hiệu khá rõ mới kiểm định được.
- Mô hình nhiều tham số (mạng nơ-ron) sẽ khớp quá mức trên cỡ mẫu này; ưu tiên mô hình ít tham số, có kiểm soát.